# Análise de Acidentes de Trânsito no Brasil

**Disciplina:** Linguagens de programação  
**Professor:** Alexandre Neves Louzada  
**Aluno:** Pedro Rigo de Oliveira

[Abrir este notebook no Google Colab](https://colab.research.google.com/github/pdroliv1/projeto_linguagem_programacao/blob/main/notebooks/analise_acidentes.ipynb)

## 1. Introdução ao problema
Este notebook analisa uma base simulada de acidentes de trânsito para identificar padrões temporais, geográficos e de gravidade. Por se tratar de dados didáticos, os resultados não representam estatísticas oficiais.

## 2. Base de dados
Cada linha contém uma observação por data, cidade e características da ocorrência. As variáveis incluem acidentes, feridos, mortes, chuva, visibilidade, veículos envolvidos e nível de gravidade.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='deep')

# Executa tanto localmente quanto no Google Colab.
caminhos_locais = [
    Path('../dados/acidentes_transito_brasil.csv'),
    Path('dados/acidentes_transito_brasil.csv')
]
fonte_github = 'https://raw.githubusercontent.com/pdroliv1/projeto_linguagem_programacao/main/dados/acidentes_transito_brasil.csv'
CAMINHO = next((caminho for caminho in caminhos_locais if caminho.exists()), fonte_github)
print(f'Fonte utilizada: {CAMINHO}')

## 3. Leitura dos dados

In [ ]:
df = pd.read_csv(CAMINHO)
df.head()

## 4. Limpeza e preparação
São verificadas duplicidades e ausências. A data é convertida para um formato temporal adequado para agregações.

In [ ]:
print('Dimensão:', df.shape)
print('Ausências:', df.isna().sum().sum())
print('Duplicidades:', df.duplicated().sum())
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df = df.dropna(subset=['data']).drop_duplicates().copy()

## 5. Engenharia de atributos
São criados período mensal, índice de vítimas e taxa de mortalidade para comparar registros com diferentes volumes de acidentes.

In [ ]:
df['periodo'] = df['data'].dt.to_period('M').astype(str)
df['indice_vitimas'] = df['feridos'] + df['mortes']
df['taxa_mortalidade'] = np.where(df['acidentes'] > 0, df['mortes'] / df['acidentes'] * 100, 0)
df[['periodo', 'indice_vitimas', 'taxa_mortalidade']].head()

## 6. Análise exploratória e 7. KPIs

In [ ]:
kpis = pd.Series({
    'Acidentes': df['acidentes'].sum(),
    'Feridos': df['feridos'].sum(),
    'Mortes': df['mortes'].sum(),
    'Taxa de mortalidade (%)': df['mortes'].sum() / df['acidentes'].sum() * 100
})
display(kpis.round(2))
resumo_regiao = df.groupby('regiao').agg(acidentes=('acidentes','sum'), mortes=('mortes','sum'), feridos=('feridos','sum'))
resumo_regiao['taxa_mortalidade'] = resumo_regiao['mortes'] / resumo_regiao['acidentes'] * 100
resumo_regiao.sort_values('acidentes', ascending=False)

## 8. Gráficos

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
resumo_regiao.sort_values('acidentes').plot.barh(y='acidentes', ax=ax, legend=False, color='#0b6e99')
ax.set(title='Acidentes por região', xlabel='Acidentes', ylabel='Região')
plt.show()

mensal = df.groupby('periodo', as_index=False)[['acidentes','mortes']].sum()
fig, ax = plt.subplots(figsize=(13, 5))
sns.lineplot(data=mensal, x='periodo', y='acidentes', marker='o', ax=ax)
ax.tick_params(axis='x', rotation=45)
ax.set(title='Evolução mensal de acidentes', xlabel='Período', ylabel='Acidentes')
plt.show()

tabela = pd.crosstab(df['visibilidade'], df['nivel_gravidade'], values=df['acidentes'], aggfunc='sum').fillna(0)
plt.figure(figsize=(8, 4))
sns.heatmap(tabela, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Gravidade por condição de visibilidade')
plt.show()

### Complemento: correlação estatística
A matriz a seguir mede a relação linear entre as variáveis quantitativas da base. Ela complementa a análise exploratória com uma funcionalidade avançada prevista na atividade.

In [ ]:
variaveis_numericas = ['acidentes', 'feridos', 'mortes', 'chuva_mm', 'veiculos_envolvidos']
correlacao = df[variaveis_numericas].corr()
plt.figure(figsize=(8, 5))
sns.heatmap(correlacao, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlação entre variáveis quantitativas')
plt.show()

## 9. Interpretação dos resultados
A comparação regional mostra a concentração do volume de acidentes no conjunto analisado. A série mensal permite observar oscilações e a matriz de calor ajuda a examinar a combinação entre visibilidade e gravidade. Para decisões reais, esses padrões precisariam ser validados com dados oficiais, cobertura da frota e exposição ao tráfego.

## 10. Conclusão
A análise transformou a base bruta em indicadores claros e visuais comparáveis. O dashboard Streamlit complementa o notebook ao permitir filtros por ano, região, UF, gravidade e tipo de acidente. A persistência em SQLite prepara o projeto para consultas e integrações futuras.